In [ ]:
# Optional: run once to install the dependency versions used for these artifacts.
from pathlib import Path
_dependency_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / "models" / "requirements.txt").exists())
%pip install -r "{_dependency_root / 'models' / 'requirements.txt'}"

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.metrics import log_loss
from sklearn.metrics import accuracy_score
import pickle

Could not save font_manager cache [Errno 13] Permission denied: 'C:\\Users\\niket\\.matplotlib\\fontlist-v3.11.0.json.matplotlib-lock'


In [2]:
from pathlib import Path
import hashlib
import json

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "processed datasets" / "X_processed.csv").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Run from the repository root or notebooks/; expected processed datasets/X_processed.csv.")
DATA_DIR = ROOT / "processed datasets"
X_processed = pd.read_csv(DATA_DIR / "X_processed.csv", index_col=0)
y_frame = pd.read_csv(DATA_DIR / "y_processed.csv", index_col=0)
if y_frame.shape[1] != 1 or not X_processed.index.equals(y_frame.index):
    raise ValueError("Expected one target column aligned with the feature row index.")
if not X_processed.index.is_unique or not X_processed.columns.is_unique:
    raise ValueError("Feature row indices and column names must be unique.")
y_processed = y_frame.iloc[:, 0]
if set(y_processed.unique()) != {0, 1}:
    raise ValueError("Expected a binary 0/1 target without missing values.")
if not all(pd.api.types.is_numeric_dtype(t) for t in X_processed.dtypes):
    raise ValueError("Expected numeric processed features.")
if not np.isfinite(X_processed.to_numpy()).all():
    raise ValueError("Features must not contain missing or infinite values.")
source_metadata = json.loads((DATA_DIR / "dataset_metadata.json").read_text(encoding="utf-8"))
if (source_metadata["num_samples"] != len(X_processed)
        or source_metadata["feature_names"] != list(X_processed.columns)
        or source_metadata["num_features"] != X_processed.shape[1]
        or source_metadata["target_name"] != y_processed.name
        or source_metadata["class_distribution"] != {str(k): int(v) for k, v in y_processed.value_counts().items()}):
    raise ValueError("The dataset metadata does not match the CSV contents.")
source_hashes = {name: hashlib.sha256((DATA_DIR / name).read_bytes()).hexdigest()
                 for name in ["X_processed.csv", "y_processed.csv", "dataset_metadata.json"]}
nonbinary_columns = [c for c in X_processed if not X_processed[c].isin([0, 1]).all()]
data_analysis = {
    "source_directory": "processed datasets",
    "source_sha256": source_hashes,
    "rows": len(X_processed), "features": X_processed.shape[1],
    "target": y_processed.name,
    "class_counts": {str(k): int(v) for k, v in y_processed.value_counts().items()},
    "positive_prevalence": float(y_processed.mean()),
    "missing_feature_values": int(X_processed.isna().sum().sum()),
    "infinite_feature_values": int(np.isinf(X_processed.to_numpy()).sum()),
    "constant_features": X_processed.columns[X_processed.nunique() <= 1].tolist(),
    "duplicate_feature_rows": int(X_processed.duplicated().sum()),
    "nonbinary_feature_statistics": X_processed[nonbinary_columns].describe().to_dict(),
}
print(f"Validated {len(X_processed):,} rows and {X_processed.shape[1]} numeric features.")
print("Class counts:", data_analysis["class_counts"])
print(f"Adverse prevalence: {y_processed.mean():.4%}")
print("Duplicate feature rows (retained):", data_analysis["duplicate_feature_rows"])
display(X_processed[nonbinary_columns].describe())

Validated 439,887 rows and 51 numeric features.
Class counts: {'0': 422300, '1': 17587}
Adverse prevalence: 3.9981%
Duplicate feature rows (retained): 832


,INIT_AGE,BMI_TCR_CLEAN,dialysis_years_at_listing,INIT_CPRA_CLEAN_MAPPED,FUNC_STATUS_PERCENT_MAPPED
count,439887.000000,439887.000000,439887.000000,439887.000000,439887.000000
mean,3.598358,28.798068,0.645674,0.286378,3.744009
std,0.373788,5.715551,0.640999,0.746853,1.388124
min,0.000000,12.410000,0.000000,0.000000,0.000000
25%,3.332205,24.650000,0.000000,0.000000,3.000000
50%,3.610918,28.510000,0.527927,0.000000,4.000000
75%,3.871201,32.810000,1.140173,0.000000,5.000000
max,4.356709,45.050000,1.843446,3.000000,6.000000


In [3]:
# Baseline A excludes gender, ethnicity, and region; no patient rows are removed.
gender_cols = [c for c in X_processed if any(s in c.upper() for s in ["GENDER", "SEX"])]
ethnic_cols = [c for c in X_processed if any(s in c.upper() for s in ["ETHCAT", "RACE"])]
region_cols = [c for c in X_processed if "REGION" in c.upper()]
excluded_columns = sorted(set(gender_cols + ethnic_cols + region_cols))
X_processed = X_processed.drop(columns=excluded_columns)
print("Excluded demographic columns:", excluded_columns)
print(f"Baseline A uses {X_processed.shape[1]} features.")

Excluded demographic columns: ['ETHCAT_1', 'ETHCAT_2', 'ETHCAT_4', 'ETHCAT_5', 'ETHCAT_6', 'ETHCAT_7', 'ETHCAT_9', 'ETHCAT_998', 'GENDER_F', 'GENDER_M', 'REGION_1', 'REGION_10', 'REGION_11', 'REGION_2', 'REGION_3', 'REGION_4', 'REGION_5', 'REGION_6', 'REGION_7', 'REGION_8', 'REGION_9']
Baseline A uses 30 features.


### Splitting the dataset

Both baselines use the same stratified 80/20 split, seed 42, and five-fold training CV. A uses 30 features after demographic exclusion; B retains all 51 features. Duplicate feature vectors are retained because identical predictors do not establish that records belong to the same patient. The CSV row indices are not patient identifiers, so patient-group separation cannot be verified here.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y_processed, test_size=0.2, random_state=42, stratify=y_processed
)
print(f"Training: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Test: {X_test.shape[0]} samples, {X_test.shape[1]} features")
print("Training class counts:", y_train.value_counts().to_dict())

Training: 351909 samples, 30 features
Test: 87978 samples, 30 features
Training class counts: {0: 337839, 1: 14070}


In [5]:
def train_test_LR(X_train, X_test, y_train, y_test, solve,  iterations, weight = 'balanced', c = 1.0):
    # Create a logistic regression model
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    model = make_pipeline(StandardScaler(), LogisticRegression(
        class_weight=weight, C=c, max_iter=iterations, solver=solve, random_state=42
    ))

    # Fit the model to the training data
    model.fit(X_train, y_train)

    #Predict the probability of the class of examples of the test data

    proba_predict = model.predict_proba(X_test)

    #Computing the log loss of the probability (proba_predict) by comparing 
    #the actual labels of the test data to the probability predictions

    logloss = log_loss(y_test, proba_predict)

    #Making predictions on the test data, to evaluate the accuracy score
    
    class_label_predict = model.predict(X_test)

    #Computing the accuracy score of the model by comparing the y_test
    #with the class label predictions generated from the step above
    
    accscore = accuracy_score(y_test, class_label_predict)
    
    return {"model": model, "predicted_proba": proba_predict, 'accuracy_score': accscore, 'log_loss': logloss, "predicted_classes": class_label_predict}

#### Pickling the Model 

In [6]:
def pickling(model, model_name):
    with open(model_name, "wb") as handle:
        pickle.dump(model, handle, protocol=pickle.HIGHEST_PROTOCOL)
    # Only load pickle files produced here or from another trusted source.
    with open(model_name, "rb") as handle:
        return pickle.load(handle)

### Automated hyperparameter optimization

GridSearchCV tests **200 settings**: 100 L2 regularization strengths (`C` from 0.0001 to 1000) and two class-weight choices (`None`, `balanced`). Five stratified folds require **1,000 validation fits**, followed by refitting the winner on the full training split. Smaller `C` applies stronger regularization. The `newton-cholesky` solver suits the large row count and small feature count. A and B use identical searches.

**Scaling rule:** `StandardScaler` is inside the pipeline. Each CV training fold learns its own means and standard deviations; the corresponding validation fold is transformed with those values. The final pipeline learns scaling from the full training split and transforms the test set with those training values. Nothing is fitted on the test set. All numeric predictors, including binary indicators, are scaled consistently with the prior fitted pipelines.

The revised CSV values are not standardized (for example, BMI retains its original scale); other upstream transformations or imputations may already have occurred. The upstream notebook still contains a full-data scaling cell, and the dataset metadata does not document preprocessing provenance. CSV values alone cannot prove every upstream transformation was leakage-free. Those steps and patient grouping remain separate validation limitations.

Average precision selects the winner because adverse outcomes are rare. The test set is used only for final reporting. No threshold is chosen from test results.

In [7]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, balanced_accuracy_score
from threadpoolctl import threadpool_limits
from joblib import parallel_config

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(solver="newton-cholesky", max_iter=1000, random_state=42)),
])
param_grid = {
    "logistic__C": np.logspace(-4, 3, 100),
    "logistic__class_weight": [None, "balanced"],
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search = GridSearchCV(
    pipeline, param_grid,
    scoring={"average_precision": "average_precision", "roc_auc": "roc_auc"},
    refit="average_precision", cv=cv, n_jobs=4, pre_dispatch=4,
    return_train_score=False, error_score="raise", verbose=1,
)
print(f"Searching {len(list(ParameterGrid(param_grid)))} configurations.")
# Avoid nested BLAS threads oversubscribing the machine during parallel CV.
with threadpool_limits(limits=1), parallel_config(backend="loky", inner_max_num_threads=1):
    search.fit(X_train, y_train)
best_model = search.best_estimator_
print("Best parameters:", search.best_params_)
print(f"Best mean CV average precision: {search.best_score_:.6f}")
cv_results = pd.DataFrame(search.cv_results_).sort_values("rank_test_average_precision")
display(cv_results[["params", "mean_test_average_precision", "std_test_average_precision", "mean_test_roc_auc"]].head(10))

Searching 200 configurations.
Fitting 5 folds for each of 200 candidates, totalling 1000 fits
Best parameters: {'logistic__C': np.float64(0.0003678379771828634), 'logistic__class_weight': None}
Best mean CV average precision: 0.080500


,params,mean_test_average_precision,std_test_average_precision,mean_test_roc_auc
16,"{'logistic__C': 0.0003678379771828634, 'logist...",0.080500,0.000820,0.677330
18,"{'logistic__C': 0.0004328761281083057, 'logist...",0.080498,0.000810,0.677365
14,"{'logistic__C': 0.00031257158496882353, 'logis...",0.080498,0.000815,0.677287
20,"{'logistic__C': 0.000509413801481638, 'logisti...",0.080491,0.000806,0.677394
12,"{'logistic__C': 0.00026560877829466864, 'logis...",0.080483,0.000821,0.677237
24,"{'logistic__C': 0.0007054802310718645, 'logist...",0.080481,0.000818,0.677440
22,"{'logistic__C': 0.0005994842503189409, 'logist...",0.080481,0.000818,0.677419
26,"{'logistic__C': 0.0008302175681319744, 'logist...",0.080475,0.000814,0.677457
28,"{'logistic__C': 0.0009770099572992247, 'logist...",0.080473,0.000817,0.677471
10,"{'logistic__C': 0.0002257019719633919, 'logist...",0.080466,0.000832,0.677178


### Final held-out evaluation

In [8]:
test_proba = best_model.predict_proba(X_test)[:, list(best_model.classes_).index(1)]
test_predictions = best_model.predict(X_test)
test_metrics = {
    "average_precision": float(average_precision_score(y_test, test_proba)),
    "roc_auc": float(roc_auc_score(y_test, test_proba)),
    "accuracy": float(accuracy_score(y_test, test_predictions)),
    "balanced_accuracy": float(balanced_accuracy_score(y_test, test_predictions)),
    "adverse_recall_at_0_5": float((test_predictions[y_test.to_numpy() == 1] == 1).mean()),
    "predicted_adverse_count_at_0_5": int((test_predictions == 1).sum()),
    "log_loss": float(log_loss(y_test, test_proba)),
}
test_confusion = confusion_matrix(y_test, test_predictions, labels=[0, 1])
print(pd.Series(test_metrics).to_string())
print(classification_report(y_test, test_predictions, zero_division=0))
print("Confusion matrix (rows=true, columns=predicted; classes 0, 1):")
print(test_confusion)
if test_metrics["predicted_adverse_count_at_0_5"] == 0:
    print("Threshold limitation: no adverse cases predicted at 0.5; accuracy reflects class imbalance.")

average_precision                 0.080930
roc_auc                           0.677869
accuracy                          0.960024
balanced_accuracy                 0.500000
adverse_recall_at_0_5             0.000000
predicted_adverse_count_at_0_5    0.000000
log_loss                          0.160207
              precision    recall  f1-score   support

           0       0.96      1.00      0.98     84461
           1       0.00      0.00      0.00      3517

    accuracy                           0.96     87978
   macro avg       0.48      0.50      0.49     87978
weighted avg       0.92      0.96      0.94     87978

Confusion matrix (rows=true, columns=predicted; classes 0, 1):
[[84461     0]
 [ 3517     0]]
Threshold limitation: no adverse cases predicted at 0.5; accuracy reflects class imbalance.


### Export and verify the best model

Save the fitted training-split pipeline, including its scaler, so this exact artifact corresponds to the held-out results. The input must be the revised processed numeric features in metadata order, **without applying an additional scaler**. Raw patient records are not supported. File hashes identify the input revision. Only fitted parameters, feature names, and aggregate summaries are exported; no row-level predictions or training records are saved.

In [9]:
import platform
import sklearn

MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)
BASELINE = "A"
artifact_prefix = f"logistic_regression_baseline_{BASELINE.lower()}"
model_path = MODEL_DIR / f"{artifact_prefix}_best.pkl"
loaded_model = pickling(best_model, model_path)
np.testing.assert_allclose(loaded_model.predict_proba(X_test), best_model.predict_proba(X_test))
metadata = {
    "baseline": BASELINE,
    "fixed_model_parameters": {"solver": "newton-cholesky", "regularization": "L2", "max_iter": 1000},
    "source_directory": "processed datasets", "source_sha256": source_hashes,
    "target": y_processed.name, "positive_class": 1,
    "feature_columns": list(X_train.columns), "excluded_columns": excluded_columns,
    "input_contract": "Revised processed numeric features in feature_columns order; pass directly to the pipeline without separate scaling. Raw records are not supported.",
    "scaling": "StandardScaler fitted within each CV training fold; winner refitted on training split only.",
    "upstream_preprocessing_provenance": "Not fully documented by source metadata; patient identifiers unavailable for grouped validation.",
    "selection_metric": "average_precision", "best_parameters": search.best_params_,
    "best_cv_average_precision": float(search.best_score_),
    "test_metrics": test_metrics, "test_confusion_matrix": test_confusion.tolist(),
    "decision_threshold": 0.5, "candidate_count": len(search.cv_results_["params"]),
    "cv_folds": 5, "test_size": 0.2, "stratified_split": True,
    "training_rows": len(X_train), "test_rows": len(X_test), "random_state": 42,
    "python_version": platform.python_version(), "sklearn_version": sklearn.__version__,
    "numpy_version": np.__version__, "pandas_version": pd.__version__,
}
(MODEL_DIR / f"{artifact_prefix}_metadata.json").write_text(
    json.dumps(metadata, indent=2), encoding="utf-8"
)
cv_results.to_csv(MODEL_DIR / f"{artifact_prefix}_cv_results.csv", index=False)
(MODEL_DIR / "processed_dataset_analysis.json").write_text(
    json.dumps(data_analysis, indent=2), encoding="utf-8"
)
print(f"Saved and reload-verified: {model_path.relative_to(ROOT)}")

Saved and reload-verified: models\logistic_regression_baseline_a_best.pkl


### Summary

**Updated Dataset Analysis**: Validated the revised CSVs and metadata in `processed datasets/`: 439,887 aligned rows, 51 numeric predictors, no missing/infinite values, and 17,587 adverse outcomes (about 4%). Retained 832 duplicate feature vectors because they are not evidence of duplicate patients. Aggregate feature ranges and SHA-256 input hashes are saved in `models/processed_dataset_analysis.json`.

**Feature Selection**: Baseline A excludes gender, ethnicity, and region, retaining 30 predictors. Both baselines use the same split, CV folds, and search settings.

**Scaling and Optimization**: StandardScaler learns parameters only from training data, within each fold and again during the winning model refit. Test data is transformed with the training scaler. GridSearchCV tests 200 combinations of L2 regularization strength and class weighting and selects the highest mean CV average precision.

**Evaluation and Persistence**: Reported held-out ranking metrics, default-threshold metrics, classification report, and confusion matrix. Exported `models/logistic_regression_baseline_a_best.pkl`, verified its probabilities after reload, and saved search results, feature order, package versions, and source hashes. The printed results below update automatically on rerun.

**Sharing and Limitations**: Files are saved locally; Git upload requires a commit and push. Load only trusted pickle files using the recorded dependency versions. No row-level records are exported. CSVs cannot establish that upstream transformations were leakage-free, and patient-group validation is unavailable without identifiers. Demographic inclusion/exclusion alone does not establish fairness. Any alternative threshold must be selected from training-only validation predictions and evaluated on fresh held-out data. This remains a research baseline.

In [10]:
# Load a trusted artifact and pass revised processed features directly to it.
with open(MODEL_DIR / f"{artifact_prefix}_metadata.json", encoding="utf-8") as handle:
    saved_metadata = json.load(handle)
with open(model_path, "rb") as handle:
    shared_model = pickle.load(handle)
example_probabilities = shared_model.predict_proba(X_test.loc[:, saved_metadata["feature_columns"]].head(5))
print("Reloaded probability shape:", example_probabilities.shape)
print(f"Baseline {BASELINE}: {len(saved_metadata['feature_columns'])} features; "
      f"CV average precision={search.best_score_:.6f}; "
      f"test average precision={test_metrics['average_precision']:.6f}; "
      f"test ROC AUC={test_metrics['roc_auc']:.6f}; "
      f"adverse recall at 0.5={test_metrics['adverse_recall_at_0_5']:.6f}.")

Reloaded probability shape: (5, 2)
Baseline A: 30 features; CV average precision=0.080500; test average precision=0.080930; test ROC AUC=0.677869; adverse recall at 0.5=0.000000.
